# Aircraft Vision Backfill — Colab

Phase 9/10 alternate path: runs the same ColQwen2 backfill as the Modal app, but on Colab's free L4 GPU. Used when the Modal workspace billing cap blocked completion.

**Required Colab Secrets** (sidebar key icon, notebook access toggled ON):
- `HF_TOKEN` — HuggingFace token (read scope) from <https://huggingface.co/settings/tokens>
- `SUPABASE_URL` — same value as `NEXT_PUBLIC_SUPABASE_URL` in `apps/web/.env.local`
- `SUPABASE_SERVICE_ROLE_KEY` — service-role key (NOT the anon key)

**Runtime:** L4 (24 GB VRAM) High-RAM works perfectly. T4 also works if you bump GPU_BATCH=1.

Total time on full 351-doc corpus: ~30–60 min on L4. Free under Colab Pro.

In [ ]:
# === Cell 1 — setup ===
# Run once per Colab session.
#
# Pin set (resolved 2026-05-09 — see vision-queue-worker.ipynb for the full
# rationale). colpali-engine 0.3.13 with transformers>=4.55 (explicit pin
# because Colab's pre-installed transformers 4.46 lacks is_flash_attn_3_available
# that colpali-engine 0.3.13 needs at import time).
!pip install -q -U "numpy>=2.0,<3.0" torch==2.4.1 torchvision==0.19.1 "transformers>=4.55,<4.58" colpali-engine==0.3.13 'accelerate>=1.0,<2.0' pdf2image pillow supabase==2.30.0 requests
!apt-get -qq install -y poppler-utils > /dev/null
import torch
import transformers
print(f"transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# === Cell 2 — secrets + Supabase ===
import os
from google.colab import userdata
from supabase import create_client

HF_TOKEN = userdata.get('HF_TOKEN')
SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_SERVICE_ROLE_KEY = userdata.get('SUPABASE_SERVICE_ROLE_KEY')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

# Sanity probe
docs_count = sb.table('documents').select('id', count='exact').limit(1).execute()
print(f"Connected. documents rows in DB: {docs_count.count}")

In [ ]:
# === Cell 3 — load ColQwen2 ===
# ~30s on cold L4 (weights ~2.2 GB).
import time
from colpali_engine.models import ColQwen2, ColQwen2Processor

MODEL_NAME = 'vidore/colqwen2-v1.0'
MODEL_LABEL = 'colqwen2'
SUMMARY_DIM = 128
MAX_PATCHES_PER_PAGE = 64

_t = time.time()
model = ColQwen2.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map='cuda', token=HF_TOKEN).eval()
processor = ColQwen2Processor.from_pretrained(MODEL_NAME, token=HF_TOKEN)
print(f"ColQwen2 loaded in {time.time()-_t:.1f}s")

In [ ]:
# === Cell 4 — helpers ===
import io, traceback
import requests
from PIL import Image
from pdf2image import convert_from_bytes

GPU_BATCH = 2  # safe on L4 (24 GB); bump to 8 on A100
DPI = 180

def embed_pil_images(images):
    """Run ColQwen2 on PIL images, return per-page summary + patch matrix."""
    results = []
    for start in range(0, len(images), GPU_BATCH):
        batch = images[start:start + GPU_BATCH]
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        with torch.no_grad():
            processed = processor.process_images(batch).to(model.device)
            page_embeds = model(**processed)
        for i in range(page_embeds.shape[0]):
            emb = page_embeds[i]
            summary = emb.mean(dim=0).to(torch.float32).cpu().tolist()
            patches_t = emb[:MAX_PATCHES_PER_PAGE].to(torch.float32).cpu()
            patches = [row.tolist() for row in patches_t]
            assert len(summary) == SUMMARY_DIM
            for r in patches: assert len(r) == SUMMARY_DIM
            results.append({'summary_vector': summary, 'patch_vectors': patches, 'patch_count': len(patches)})
    return results

def get_or_insert_vision_page(org_id, doc_id, page_n, image_path):
    """Idempotent insert respecting partial unique index. On 23505 fetch existing + reset to 'embedding'."""
    try:
        r = sb.table('vision_pages').insert({
            'organization_id': org_id, 'source_document_id': doc_id,
            'page_number': page_n, 'page_image_path': image_path,
            'status': 'embedding',
        }).execute()
        return r.data[0]['id']
    except Exception as e:
        if '23505' in str(e) or 'duplicate' in str(e):
            existing = sb.table('vision_pages').select('id').eq('organization_id', org_id).eq('source_document_id', doc_id).eq('page_number', page_n).is_('deleted_at', 'null').limit(1).execute()
            if existing.data:
                vid = existing.data[0]['id']
                sb.table('vision_pages').update({'status': 'embedding', 'error_message': None}).eq('id', vid).execute()
                return vid
        raise

def upload_page_image(org_id, doc_id, page_n, pil_img):
    image_path = f"{org_id}/{doc_id}/page_{page_n}.png"
    buf = io.BytesIO()
    pil_img.save(buf, format='PNG')
    try:
        sb.storage.from_('vision-pages').upload(image_path, buf.getvalue(), {'contentType': 'image/png', 'upsert': 'true'})
    except Exception as e:
        if 'already exists' not in str(e).lower():
            raise
    return image_path

def backfill_one_doc(doc):
    org_id = doc['organization_id']
    doc_id = doc['id']
    file_path = doc.get('file_path')
    if not file_path:
        return 0, 0, ['no file_path']
    try:
        pdf_bytes = sb.storage.from_('documents').download(file_path)
        if not pdf_bytes:
            return 0, 0, ['empty pdf']
        pil_pages = convert_from_bytes(pdf_bytes, dpi=DPI, fmt='png')
    except Exception as e:
        return 0, 0, [f'pdf: {e}']

    vision_page_ids = []
    for n, img in enumerate(pil_pages):
        try:
            image_path = upload_page_image(org_id, doc_id, n, img)
            vid = get_or_insert_vision_page(org_id, doc_id, n, image_path)
            vision_page_ids.append(vid)
        except Exception as e:
            return 0, len(pil_pages), [f'page {n} setup: {e}']

    try:
        embeds = embed_pil_images(pil_pages)
    except Exception as e:
        for vid in vision_page_ids:
            sb.table('vision_pages').update({'status': 'failed', 'error_message': f'gpu: {e}'}).eq('id', vid).execute()
        return 0, len(pil_pages), [f'gpu: {e}']

    indexed, failed, errs = 0, 0, []
    for n, vid in enumerate(vision_page_ids):
        er = embeds[n]
        try:
            sb.table('vision_embeddings').insert({
                'organization_id': org_id, 'vision_page_id': vid,
                'model_used': MODEL_LABEL, 'embedding_dim': SUMMARY_DIM,
                'summary_vector': er['summary_vector'],
                'patch_vectors': {'patches': er['patch_vectors']},
                'patch_count': er['patch_count'],
            }).execute()
            sb.table('vision_pages').update({
                'status': 'indexed', 'vision_index_id': vid,
                'vision_model': MODEL_LABEL, 'embedded_at': 'now()',
            }).eq('id', vid).execute()
            indexed += 1
        except Exception as e:
            sb.table('vision_pages').update({'status': 'failed', 'error_message': str(e)[:1000]}).eq('id', vid).execute()
            failed += 1
            errs.append(f'page {n} insert: {e}')
    return indexed, failed, errs

print("Helpers loaded.")

In [ ]:
# === Cell 5 — RUN ===
# Walk every doc that doesn't have any indexed vision_pages yet.
# Set MAX_DOCS=5 first to dry-run; flip to None for the full run.

MAX_DOCS = None
SLEEP_BETWEEN_DOCS = 0

import time

all_docs_resp = sb.table('documents').select('id, organization_id, file_path, file_name, page_count').is_('deleted_at', 'null').limit(2000).execute()
all_docs = [d for d in (all_docs_resp.data or []) if d.get('file_path')]

indexed_docs_resp = sb.table('vision_pages').select('source_document_id').eq('status', 'indexed').limit(2000).execute()
indexed_doc_ids = {r['source_document_id'] for r in (indexed_docs_resp.data or []) if r.get('source_document_id')}

todo = [d for d in all_docs if d['id'] not in indexed_doc_ids]
todo.sort(key=lambda d: (d.get('page_count') or 999, d['file_name'] or ''))
if MAX_DOCS:
    todo = todo[:MAX_DOCS]

print(f"Total docs in DB: {len(all_docs)}, already-indexed: {len(indexed_doc_ids)}, todo: {len(todo)}")

started_at = time.time()
total_indexed = 0
total_failed = 0
errors_log = []

for i, doc in enumerate(todo, 1):
    t0 = time.time()
    indexed, failed, errs = backfill_one_doc(doc)
    total_indexed += indexed
    total_failed += failed
    if errs:
        errors_log.extend(errs[:1])
    elapsed = time.time() - t0
    cum_elapsed = (time.time() - started_at) / 60
    pct = i / len(todo) * 100
    print(f"[{i}/{len(todo)} {pct:.1f}%] {doc['file_name'][:50]}: +{indexed} -{failed} ({elapsed:.1f}s)  | cum {total_indexed}/{total_failed}, {cum_elapsed:.1f}min")
    if SLEEP_BETWEEN_DOCS:
        time.sleep(SLEEP_BETWEEN_DOCS)

print("")
print("=" * 60)
print(f"DONE: indexed {total_indexed} pages, failed {total_failed} pages, {(time.time()-started_at)/60:.1f} min")
if errors_log:
    print(f"\nFirst 5 errors:")
    for e in errors_log[:5]:
        print(f"  {e[:200]}")